# SmartCart - Notebook 1: Data Preprocessing

This notebook prepares the raw e-commerce data for downstream recommendation and pattern-mining tasks.

## Objectives
- Load interaction and catalog datasets
- Standardize and clean key columns
- Validate ratings and timestamps
- Build a user-item matrix for collaborative filtering
- Aggregate user behavior by product category

## Inputs
- `data/raw/ecommerce_user_data.csv`
- `data/raw/product_details.csv`

The loading logic also supports `data/` as a fallback path for compatibility.

In [1]:
# Imports and display settings
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [2]:
# Resolve file paths with raw-first, fallback-to-data strategy
cwd = Path.cwd()
search_roots = [cwd, cwd.parent]
candidate_dirs = []
for base_root in search_roots:
    candidate_dirs.extend([base_root / 'data' / 'raw', base_root / 'data'])

user_path = None
product_path = None
for base in candidate_dirs:
    u = base / 'ecommerce_user_data.csv'
    p = base / 'product_details.csv'
    if u.exists() and p.exists():
        user_path = u
        product_path = p
        break

if user_path is None or product_path is None:
    raise FileNotFoundError(
        'Could not find input CSV files in data/raw or data. '
        f'Current working directory: {cwd}'
    )

print('Using files:')
print('-', user_path)
print('-', product_path)

Using files:
- /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/raw/ecommerce_user_data.csv
- /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/raw/product_details.csv


In [3]:
# Load and clean base datasets
user_data = pd.read_csv(user_path)
product_data = pd.read_csv(product_path)

# Standardize column names
user_data.columns = user_data.columns.str.strip()
product_data.columns = product_data.columns.str.strip()

# Remove duplicate records
user_data = user_data.drop_duplicates()
product_data = product_data.drop_duplicates(subset=['ProductID'])

# Clean text fields
for col in ['UserID', 'ProductID', 'Category']:
    if col in user_data.columns:
        user_data[col] = user_data[col].astype(str).str.strip()

for col in ['ProductID', 'Category', 'ProductName']:
    if col in product_data.columns:
        product_data[col] = product_data[col].astype(str).str.strip()

# Coerce ratings and keep valid range
user_data['Rating'] = pd.to_numeric(user_data['Rating'], errors='coerce')
user_data = user_data[user_data['Rating'].between(1, 5, inclusive='both')]

# Parse timestamp safely
user_data['Timestamp'] = pd.to_datetime(user_data['Timestamp'], errors='coerce')

# Fill missing interaction category from product catalog
product_category_map = product_data.set_index('ProductID')['Category']
user_data['Category'] = user_data['Category'].replace({'': pd.NA, 'nan': pd.NA})
user_data['Category'] = user_data['Category'].fillna(user_data['ProductID'].map(product_category_map))

# Drop unusable records
user_data = user_data.dropna(subset=['UserID', 'ProductID', 'Rating', 'Category'])

# Keep stable ordering for reproducibility
user_data = user_data.sort_values(['UserID', 'Timestamp', 'ProductID']).reset_index(drop=True)

print('Cleaned user_data shape:', user_data.shape)
print('Product catalog shape:', product_data.shape)
display(user_data.head())
display(product_data.head())

Cleaned user_data shape: (724, 5)
Product catalog shape: (100, 3)


,UserID,ProductID,Rating,Timestamp,Category
0,U000,P0020,1,2024-09-02,Home
1,U000,P0071,3,2024-09-02,Toys
2,U000,P0007,3,2024-09-03,Books
3,U000,P0044,2,2024-09-06,Books
4,U000,P0047,3,2024-09-06,Clothing


,ProductID,ProductName,Category
0,P0000,Toys Item 0,Clothing
1,P0001,Clothing Item 1,Electronics
2,P0002,Books Item 2,Electronics
3,P0003,Clothing Item 3,Electronics
4,P0004,Clothing Item 4,Electronics


In [4]:
# Create user-item matrix
user_item_matrix = user_data.pivot_table(
    index='UserID',
    columns='ProductID',
    values='Rating',
    aggfunc='mean'
)

# Fill unrated items with 0 for collaborative filtering workflows
user_item_matrix_filled = user_item_matrix.fillna(0)

print('User-item matrix shape:', user_item_matrix_filled.shape)
user_item_matrix_filled.head()
# Aggregate user behavior by category
user_category_agg = (
    user_data
    .groupby(['UserID', 'Category'], as_index=False)
    .agg(
        TotalInteractions=('Rating', 'count'),
        AverageRating=('Rating', 'mean'),
        LastInteraction=('Timestamp', 'max')
    )
    .sort_values(['UserID', 'TotalInteractions'], ascending=[True, False])
    .reset_index(drop=True)
)

user_category_agg.head()

User-item matrix shape: (50, 100)


,UserID,Category,TotalInteractions,AverageRating,LastInteraction
0,U000,Books,6,3.666667,2024-10-18
1,U000,Toys,6,3.500000,2024-10-18
2,U000,Clothing,3,1.666667,2024-10-14
3,U000,Electronics,3,3.666667,2024-10-28
4,U000,Home,2,1.000000,2024-09-15


In [5]:
# Export cleaned artifacts for downstream notebooks
if user_path.parent.name == 'raw':
    project_root = user_path.parents[2]
else:
    project_root = user_path.parents[1]

processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

user_data.to_csv(processed_dir / 'user_data_clean.csv', index=False)
product_data.to_csv(processed_dir / 'product_data_clean.csv', index=False)
user_item_matrix_filled.to_csv(processed_dir / 'user_item_matrix_filled.csv')
user_category_agg.to_csv(processed_dir / 'user_category_agg.csv', index=False)

print('Saved processed files to:', processed_dir)
print('-', processed_dir / 'user_data_clean.csv')
print('-', processed_dir / 'product_data_clean.csv')
print('-', processed_dir / 'user_item_matrix_filled.csv')
print('-', processed_dir / 'user_category_agg.csv')

Saved processed files to: /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/processed
- /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/processed/user_data_clean.csv
- /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/processed/product_data_clean.csv
- /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/processed/user_item_matrix_filled.csv
- /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/processed/user_category_agg.csv


## Outputs Generated
- `user_data`: cleaned interaction-level dataset
- `product_data`: deduplicated product catalog
- `user_item_matrix_filled`: matrix used for collaborative filtering
- `user_category_agg`: category-level behavioral summary by user

### Files exported to `data/processed/`
- `user_data_clean.csv`
- `product_data_clean.csv`
- `user_item_matrix_filled.csv`
- `user_category_agg.csv`

These exported artifacts can be reused directly in Notebook 2 and Notebook 4 without repeating preprocessing.